In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [3]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# process the data
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [5]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [6]:
# Onehot encode the "Geography" column
onehot_encoder_geography = OneHotEncoder(handle_unknown='ignore')
geography_encoded = onehot_encoder_geography.fit_transform(data[["Geography"]]).toarray()
geography_encoded_df = pd.DataFrame(geography_encoded, columns=onehot_encoder_geography.get_feature_names_out(["Geography"]))
geography_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [7]:
# combine onehot encoded columns with the original dataframe
data = pd.concat([data.drop("Geography",axis=1),geography_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [18]:
# Split the data into features and target variable
X = data.drop("EstimatedSalary",axis=1)
Y = data["EstimatedSalary"]

In [19]:
# Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Save the encoders and scaler for future use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
    
with open('onehot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geography, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [10]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [13]:
# Build the model
model = Sequential([
  Dense(64, activation='relu', input_shape=(X.shape[1],)),
  Dense(32, activation='relu'),
  Dense(1)
])

model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

# Setup tensorboard 
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [15]:
# Setup early stopping
early_stopping_callabck = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [20]:
# Train the model
history = model.fit(
  X_train, Y_train,
  validation_data=(X_test, Y_test),
  epochs=100,
  callbacks=[early_stopping_callabck, tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 4s 5ms/step - loss: 71318.5625 - mae: 71318.5625 - val_loss: 66795.8516 - val_mae: 66795.8516
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 65105.9219 - mae: 65105.9219 - val_loss: 60244.0664 - val_mae: 60244.0664
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 56572.9961 - mae: 56572.9961 - val_loss: 52159.5156 - val_mae: 52159.5156
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 52142.4023 - mae: 52142.4023 - val_loss: 50806.0273 - val_mae: 50806.0273
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 51628.7930 - mae: 51628.7930 - val_loss: 50615.5781 - val_mae: 50615.5781
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 51510.3711 - mae: 51510.3711 - val_loss: 51128.3477 - val_mae: 51128.3477
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 51324.0586 - mae: 51324.0586 

In [27]:
# Add the env's Scripts folder to PATH so %tensorboard can find tensorboard.exe
import os, sys
scripts = os.path.join(os.path.dirname(sys.executable), "Scripts")
os.environ["PATH"] = scripts + os.pathsep + os.environ["PATH"]
os.environ["TENSORBOARD_BINARY"] = os.path.join(scripts, "tensorboard.exe")

# Load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [30]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 13900), started 0:00:20 ago. (Use '!kill 13900' to kill it.)

In [31]:
# Evaluate the model on the test set
test_loss,test_mae = model.evaluate(X_test,Y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

63/63 [==============================] - 0s 2ms/step - loss: 50316.9258 - mae: 50316.9258
Test Loss: 50316.92578125, Test MAE: 50316.92578125


In [32]:
model.save('ann_regression_model.h5')

d:\Python\ANN-Implementation\ANN-Regression\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
